# Dispersion-per-band diagnostics -- Simbad-confirmed stable stars

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Universite Paris-Saclay
- **Created:** 2026-07-16

## Goal

In `07_FindVisitsInAllbands/usdf_butler/02_ReadLSSTCamSourcesInAllbands.ipynb`
the per-object photometric scatter `mmag_meas` is found to exceed the
photon-noise-only expectation `mmag_phot` for a *large, unfiltered* sample of
sources (mag 17-19.5 in the DDFs). That sample implicitly assumes most stars
are photometrically stable, which is not guaranteed -- some excess scatter
could simply be intrinsically variable stars contaminating the sample.

This notebook re-runs the **same family of diagnostic plots** (2D histogram
of dispersion vs. magnitude with the photon-noise floor overlaid, boxplots,
violin plots) but on the small sample of stars whose stability is
**guaranteed by Simbad** (selected in `05_FindSources/usdf_butler`, N~30
stars). If the excess scatter seen in the large sample persists here, it is
much less likely to be a contamination-by-variables effect and more likely a
genuine instrumental/atmospheric floor (e.g. calibration residuals,
photometric-condition variability) present even for bona fide stable stars.

**Caveat:** with only ~30 stars split six ways in band and further split by
magnitude bin, several panels/bins below will be sparse (small-N). This
notebook is meant as a **cross-check**, to be revisited once the larger
Simbad-confirmed sample (200+ stars) currently being extracted in
`05b_FindSourcesAllSpectypes/usdf_butler` is available.

## Input data

Per-visit light-curve table written by `02_MergeLCsourceswithMJD.ipynb`
(read the same way as in `03_PlotLCwithMJD.ipynb`):
`data_MergeVisits_02_out/all_stars_lightcurves_mjd.csv`, one row per
(star, visit), with columns including `simbad_id`, `band`, `psfFlux`,
`psfFluxErr`, `ra`, `dec`.

This notebook first collapses that per-visit table into a **per-(star,
band) summary table** (`df_all`) with the same columns as the
`objectstats_band_<band>.parquet` files consumed by
`02_ReadLSSTCamSourcesInAllbands.ipynb` (`object_id`, `n_visits`, `ra`,
`dec`, `mag_median`, `mmag_meas`, `mmag_phot`, ...), so that the plotting
code below mirrors that notebook as closely as possible.

## 1. Imports

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

from scipy.optimize import curve_fit

In [ ]:
# view all contents of pandas tables
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> %matplotlib inline")

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging is configured and working in the notebook!")

## 3. Configuration

**Edit only this cell** to point to the input light-curve file (same file
read by `03_PlotLCwithMJD.ipynb`) and to adjust the small-N-aware binning
parameters used below.

In [ ]:
# -- Notebook tag ------------------------------------------------------------
NB_TAG = "DispersionPerBand_11"

# -- Input: per-visit merged LC file from notebook 02 ------------------------
DIR_DATA_IN = "./data_MergeVisits_02_out"
LC_CSV = os.path.join(DIR_DATA_IN, "all_stars_lightcurves_mjd.csv")

# -- Output figures ------------------------------------------------------------
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# -- Photometric columns in the per-visit table -------------------------------
FLUX_COL = "psfFlux"
FLUX_ERR_COL = "psfFluxErr"
BAND_COL = "band"
STAR_COL = "simbad_id"

# -- AB magnitude zero point for fluxes expressed in nJy ----------------------
ABMAG_ZP_NJY = 31.4

# -- All LSST bands, in the standard display order ----------------------------
BANDS = ["u", "g", "r", "i", "z", "y"]
BANDS_CMAP = {"u": "Purples", "g": "Greens", "r": "Reds", "i": "YlOrBr", "z": "pink_r", "y": "bone_r"}
BANDS_COLOR = {
    "u": "blueviolet",
    "g": "limegreen",
    "r": "red",
    "i": "darkorange",
    "z": "chocolate",
    "y": "saddlebrown",
}

# -- Minimum number of good visits to keep a (star, band) row in df_all -------
MIN_VISITS_PER_STARBAND = 5

# -- Small-N-aware settings (this sample is only ~30 stars, unlike the large
#    unfiltered sample in 07_FindVisitsInAllbands): wider magnitude bins and a
#    lower minimum-points-per-bin threshold than the defaults used there.
MAG_BIN_WIDTH = 1.0  # mag, vs. 0.5 in 07_FindVisitsInAllbands/.../02_...ipynb
MIN_PTS_PER_BIN = 3  # vs. 5 in 07_FindVisitsInAllbands/.../02_...ipynb

log.info(f"Reading per-visit light curves from '{LC_CSV}'")

## 4. Helper functions

In [ ]:
# -- savefig: PDF + PNG -------------------------------------------------------
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)

In [ ]:
# -- helpers for statistics ----------------------------------------------------
def sigma_iqr(x):
    """Robust scatter estimator: interquartile range rescaled so that it
    matches the standard deviation for a pure Gaussian distribution."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    q25, q75 = np.percentile(x, [25, 75])
    return (q75 - q25) / 1.3489795


def flux_to_mag(flux_njy):
    """AB magnitude from a flux expressed in nJy, using ABMAG_ZP_NJY."""
    flux_njy = np.asarray(flux_njy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = ABMAG_ZP_NJY - 2.5 * np.log10(flux_njy)
    return mag


def sigma_nJy_to_mmag(sigma_nJy, median_nJy):
    """Convert a flux scatter sigma (nJy) to mmag using the linear
    approximation delta_mag ~= (2.5/ln10) * (sigma/median)."""
    if not np.isfinite(median_nJy) or median_nJy <= 0 or not np.isfinite(sigma_nJy) or sigma_nJy < 0:
        return np.nan
    return 1000.0 * (2.5 / np.log(10)) * (sigma_nJy / abs(median_nJy))

In [ ]:
# -- helpers for fitting ---------------------------------------------------
def gaussian(x, amp, mu, sigma):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def fit_gaussian_to_hist(data, n_bins=40, x_range=None):
    """Fit a Gaussian to the histogram of `data`.

    Returns a dict with keys `mu`, `sigma`, `amp`, `edges` on success, or
    None if the fit could not be performed (too few points or curve_fit
    failure).
    """
    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]
    if len(data) < 10:
        return None

    if x_range is None:
        x_range = (0.0, np.nanpercentile(data, 99.0))

    counts, edges = np.histogram(data, bins=n_bins, range=x_range)
    centers = 0.5 * (edges[:-1] + edges[1:])

    siqr = sigma_iqr(data)
    p0 = [max(counts.max(), 1.0), np.median(data), siqr if siqr > 0 else np.std(data)]

    try:
        popt, _ = curve_fit(gaussian, centers, counts, p0=p0, maxfev=5000)
    except Exception:
        return None

    amp, mu, sigma = popt
    return {"amp": amp, "mu": mu, "sigma": abs(sigma), "edges": edges}

In [ ]:
def median_curve_vs_mag(
    mag, y, mag_min, mag_max, n_bins=40, min_pts=3, smooth=True, smooth_window=7, smooth_poly=2
):
    """Robust, smooth curve of the median of `y` vs `mag`.

    Strategy: bin in magnitude with `n_bins` (finer than the diagnostic
    magnitude bins used elsewhere in the notebook) and take the median of
    `y` in each bin -- this is already robust to outliers and needs no
    assumption on the functional shape of `y(mag)`. Since `mmag_phot`
    (photon-noise-only expectation) is intrinsically a smooth, monotonic
    function of magnitude, a light Savitzky-Golay pass on the binned
    medians removes residual bin-to-bin noise (small-N bins) without
    biasing the trend or flattening real curvature, unlike a boxcar/moving
    average. Returns (mag_centers, y_smooth), keeping only bins with at
    least `min_pts` points.
    """
    mag = np.asarray(mag, dtype=float)
    y = np.asarray(y, dtype=float)
    sel = np.isfinite(mag) & np.isfinite(y) & (y > 0)
    mag, y = mag[sel], y[sel]

    edges = np.linspace(mag_min, mag_max, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    meds = np.full(n_bins, np.nan)
    for k in range(n_bins):
        lo, hi = edges[k], edges[k + 1]
        in_bin = (mag >= lo) & (mag <= hi if k == n_bins - 1 else mag < hi)
        vals = y[in_bin]
        if len(vals) >= min_pts:
            meds[k] = np.median(vals)

    valid = np.isfinite(meds)
    centers, meds = centers[valid], meds[valid]

    if smooth and len(meds) >= 5:
        from scipy.signal import savgol_filter

        w = min(smooth_window, len(meds) - (1 - len(meds) % 2))
        if w % 2 == 0:
            w -= 1
        if w >= 3:
            meds = savgol_filter(meds, window_length=w, polyorder=min(smooth_poly, w - 1))

    return centers, meds

## 5. Read the per-visit light curves and build the per-(star, band)
summary table

`df_all` is built to match the columns produced by `summarize_objects()` in
`07_FindVisitsInAllbands`: one row per (star, band), with `mag_median` (AB
mag of the median flux), `mmag_meas` (measured scatter, robust
sigma_IQR-based), and `mmag_phot` (photon-noise-only expectation, from the
median reported `psfFluxErr`).

In [ ]:
df_lc = pd.read_csv(LC_CSV)
log.info(f"Read {len(df_lc)} per-visit rows for {df_lc[STAR_COL].nunique()} stars from {LC_CSV}")
df_lc.head()

In [ ]:
rows = []

for (sid, band), grp in df_lc.groupby([STAR_COL, BAND_COL]):
    flux = grp[FLUX_COL].to_numpy(dtype=float)
    err = grp[FLUX_ERR_COL].to_numpy(dtype=float)
    sel = np.isfinite(flux) & np.isfinite(err) & (err > 0)
    flux, err = flux[sel], err[sel]

    n_visits = len(flux)
    if n_visits < MIN_VISITS_PER_STARBAND:
        continue

    flux_median = np.median(flux)
    if not np.isfinite(flux_median) or flux_median <= 0:
        continue

    flux_siqr = sigma_iqr(flux)
    err_median = np.median(err)

    rows.append(
        {
            "object_id": sid,
            "band": band,
            "n_visits": n_visits,
            "ra": grp["ra"].mean(),
            "dec": grp["dec"].mean(),
            "flux_median": flux_median,
            "flux_sigma_iqr": flux_siqr,
            "mag_median": flux_to_mag(flux_median),
            "sigmaF_over_F_meas": flux_siqr / flux_median,
            "sigmaF_over_F_phot": err_median / flux_median,
            "mmag_meas": sigma_nJy_to_mmag(flux_siqr, flux_median),
            "mmag_phot": sigma_nJy_to_mmag(err_median, flux_median),
        }
    )

df_all = pd.DataFrame(rows)
log.info(
    f"Built per-(star, band) summary table: {len(df_all)} rows, "
    f"{df_all['object_id'].nunique()} distinct stars, "
    f">= {MIN_VISITS_PER_STARBAND} visits/band required."
)
df_all.head()

In [ ]:
df_all.groupby("band")["mmag_meas"].describe()[["count", "mean", "50%", "std"]]

In [ ]:
# -- Magnitude window, auto-determined from this sample (rounded to the
# nearest 0.5 mag), used for plot titles/limits below.
MAG_MIN = np.floor(df_all["mag_median"].min() * 2) / 2
MAG_MAX = np.ceil(df_all["mag_median"].max() * 2) / 2
log.info(f"Magnitude range spanned by the sample: {MAG_MIN:.1f} < mag < {MAG_MAX:.1f}")

## 6. Plots

### 6.1  Relative photometric scatter (mmag) per band -- boxplot

Boxplot of `mmag_meas` grouped by band, in the standard LSST band order
`u, g, r, i, z, y`. Same figure as section 8.1 of
`02_ReadLSSTCamSourcesInAllbands.ipynb`, for the Simbad-confirmed stable
stars only.

In [ ]:
band_order = [b for b in BANDS if b in df_all["band"].unique()]

data = [df_all.loc[df_all["band"] == b, "mmag_meas"].dropna().to_numpy() for b in band_order]

fig, ax = plt.subplots(figsize=(7, 5))

flier_props = dict(
    marker="o",
    markersize=6,
    markerfacecolor="none",
    markeredgecolor="lightgray",
    markeredgewidth=1.0,
    alpha=0.6,
)

ax.boxplot(data, tick_labels=band_order, showfliers=True, flierprops=flier_props)

ax.set_xlabel("Band")
ax.set_ylabel(r"$\sigma_F/F$ (mmag)")
ax.set_title(
    f"Relative PSF-flux scatter, Simbad-confirmed stable stars ({MAG_MIN:.1f} < mag < {MAG_MAX:.1f})"
)
ax.grid(True, alpha=0.3)

for idx, d in enumerate(data, start=1):
    ax.text(
        idx,
        0.95,
        f"n={len(d)}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=9,
        color="darkgray",
    )

plt.tight_layout()
savefig(fig, "mmag_scatter_stablestars_perband")
plt.show()

### 6.2  Dispersion vs. magnitude, 2D histogram per band

`2 x 3` grid of 2D histograms of `mmag_meas` (dispersion, log-binned) vs.
`mag_median` (magnitude, linear-binned), one panel per band, order
`u, g, r, i, z, y`. The black dashed line is the smoothed photon-noise-only
floor (`mmag_phot`, via `median_curve_vs_mag`). Same figure as section 8.2
of `02_ReadLSSTCamSourcesInAllbands.ipynb`.

With only ~30 stars, expect each panel to show a **sparse point cloud**
rather than a dense 2D histogram -- read the colorbar counts carefully.

In [ ]:
def plot_disp_vs_mag(
    ax, mag, disp, band, mag_phot_curve=None, mag_min=MAG_MIN, mag_max=MAG_MAX, n_xbins=20, n_ybins=20
):
    """2D histogram of dispersion (mmag_meas, log-binned) vs magnitude
    (linear-binned) on a single axis, with a log-scaled y-axis so the
    high-dispersion tail is visible. If `mag_phot_curve` is given as a
    `(mag_centers, mmag_phot_smooth)` tuple, it is overlaid as a black
    dashed line: the photon-noise-only floor that the bulk of the 2D
    histogram is expected to sit close to, with any real excess (e.g.
    atmospheric transparency variations) showing up as a population lifted
    above this line.
    """
    mag = np.asarray(mag, dtype=float)
    disp = np.asarray(disp, dtype=float)
    sel = np.isfinite(mag) & np.isfinite(disp) & (disp > 0)
    mag, disp = mag[sel], disp[sel]

    if len(disp) == 0:
        ax.set_title(f"band {band} (no data)")
        return None

    xedges = np.linspace(mag_min, mag_max, n_xbins + 1)

    ylo = max(np.nanpercentile(disp, 0.5) * 0.8, disp[disp > 0].min())
    yhi = np.nanpercentile(disp, 99.5) * 1.5
    if not np.isfinite(ylo) or not np.isfinite(yhi) or ylo <= 0 or yhi <= ylo:
        ylo, yhi = disp.min() * 0.8, disp.max() * 1.2
    yedges = np.logspace(np.log10(ylo), np.log10(yhi), n_ybins + 1)

    h, xe, ye = np.histogram2d(mag, disp, bins=[xedges, yedges])
    mesh = ax.pcolormesh(xe, ye, h.T, norm=LogNorm(vmin=1, vmax=max(h.max(), 1)), cmap=BANDS_CMAP[band])
    ax.set_yscale("log")
    ax.set_xlim(mag_min, mag_max)

    if mag_phot_curve is not None and len(mag_phot_curve[0]) > 0:
        curve_mag, curve_y = mag_phot_curve
        ax.plot(
            curve_mag,
            curve_y,
            color="black",
            lw=2.0,
            ls="--",
            label="photon-noise floor (mmag_phot)",
        )
        ax.legend(loc="upper left", fontsize=8, frameon=True)

    ax.set_title(f"band {band}  (N={len(disp)})")
    ax.grid(True, alpha=0.2, which="both")
    return mesh


fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)

band_grid_order = ["u", "g", "r", "i", "z", "y"]

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue
    sub = df_all.loc[df_all["band"] == band]

    phot_curve = median_curve_vs_mag(
        sub["mag_median"], sub["mmag_phot"], MAG_MIN, MAG_MAX, n_bins=20, min_pts=MIN_PTS_PER_BIN
    )

    mesh = plot_disp_vs_mag(ax, sub["mag_median"], sub["mmag_meas"], band, mag_phot_curve=phot_curve)
    if mesh is not None:
        fig.colorbar(mesh, ax=ax, label="N objects")

for ax in axes[1, :]:
    ax.set_xlabel("median magnitude")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")

fig.suptitle(
    f"Photometric scatter vs. magnitude per band, Simbad-confirmed stable stars\n"
    "(dashed line: photon-noise floor mmag_phot, smoothed median per fine mag bin)",
    y=1.02,
    fontsize=16,
)
plt.tight_layout()
savefig(fig, "mmag_vs_mag_hist2d_per_band_stablestars")
plt.show()

### 6.2bis  Control plot: `median_curve_vs_mag` fit check
(`mmag_phot` vs `mag_median`)

Sanity-check figure for the smoothed curve overlaid above: raw `mmag_phot`
vs `mag_median` scatter together with the `median_curve_vs_mag` fit, per
band. Same as section 8.2bis of `02_ReadLSSTCamSourcesInAllbands.ipynb`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue

    sub = df_all.loc[df_all["band"] == band]
    mag = sub["mag_median"].to_numpy(dtype=float)
    phot = sub["mmag_phot"].to_numpy(dtype=float)
    sel = np.isfinite(mag) & np.isfinite(phot) & (phot > 0)
    mag, phot = mag[sel], phot[sel]

    ax.scatter(mag, phot, s=16, color=BANDS_COLOR[band], alpha=0.6, label="mmag_phot (data)")

    curve_mag, curve_y = median_curve_vs_mag(mag, phot, MAG_MIN, MAG_MAX, n_bins=20, min_pts=MIN_PTS_PER_BIN)
    if len(curve_mag) > 0:
        ax.plot(curve_mag, curve_y, color="black", lw=2.0, ls="--", label="median_curve_vs_mag fit")

    ax.set_yscale("log")
    ax.set_xlim(MAG_MIN, MAG_MAX)
    ax.set_title(f"band {band}  (N={len(phot)})")
    ax.grid(True, alpha=0.2, which="both")
    ax.legend(loc="upper left", fontsize=8, frameon=True)

for ax in axes[1, :]:
    ax.set_xlabel("median magnitude")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), photon-noise-only")

fig.suptitle(
    "Control plot: median_curve_vs_mag fit vs. raw mmag_phot data, per band -- stable stars",
    y=1.0,
    fontsize=16,
)
plt.tight_layout()
savefig(fig, "mmag_phot_vs_mag_fitcheck_per_band_stablestars")
plt.show()

### 6.3  Median dispersion vs. magnitude, error-bar plot (all bands,
single axis)

One point per (band, magnitude bin): `x = mag_center +/- mag_halfwidth`,
`y = median(mmag_meas) +/- sigma_IQR(mmag_meas)`. The dashed line per band
is the smoothed photon-noise-only floor. Same as section 8.4 of
`02_ReadLSSTCamSourcesInAllbands.ipynb`, with `MAG_BIN_WIDTH` and
`MIN_PTS_PER_BIN` widened/lowered for this small sample (see Configuration
cell).

In [ ]:
mag_edges = np.arange(MAG_MIN, MAG_MAX + 1e-9, MAG_BIN_WIDTH)
if mag_edges[-1] < MAG_MAX - 1e-9:
    mag_edges = np.append(mag_edges, MAG_MAX)
n_mag_bins = len(mag_edges) - 1
mag_bin_labels = [f"{mag_edges[k]:.1f}-{mag_edges[k + 1]:.1f}" for k in range(n_mag_bins)]

log.info(f"{n_mag_bins} magnitude bins of width {MAG_BIN_WIDTH} mag: {np.round(mag_edges, 2).tolist()}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for band in band_grid_order:
    if band not in df_all["band"].unique():
        continue

    sub_band = df_all.loc[df_all["band"] == band]

    mag_centers, mag_halfwidths, medians, siqrs = [], [], [], []

    for k in range(n_mag_bins):
        lo, hi = mag_edges[k], mag_edges[k + 1]
        last_bin = k == n_mag_bins - 1
        sel = sub_band["mag_median"].between(lo, hi, inclusive="both" if last_bin else "left")
        disp = sub_band.loc[sel, "mmag_meas"].dropna().to_numpy()
        disp = disp[np.isfinite(disp) & (disp > 0)]

        if len(disp) < MIN_PTS_PER_BIN:
            continue

        mag_centers.append(np.mean([lo, hi]))
        mag_halfwidths.append((hi - lo) / 2.0)
        medians.append(np.median(disp))
        siqrs.append(sigma_iqr(disp))

    if len(mag_centers) == 0:
        continue

    ax.errorbar(
        mag_centers,
        medians,
        xerr=mag_halfwidths,
        yerr=siqrs,
        fmt="o",
        color=BANDS_COLOR[band],
        ecolor=BANDS_COLOR[band],
        elinewidth=1.2,
        capsize=3,
        markersize=10,
        label=band,
    )

    curve_mag, curve_y = median_curve_vs_mag(
        sub_band["mag_median"], sub_band["mmag_phot"], MAG_MIN, MAG_MAX, n_bins=20, min_pts=MIN_PTS_PER_BIN
    )
    if len(curve_mag) > 0:
        ax.plot(curve_mag, curve_y, color=BANDS_COLOR[band], lw=1.3, ls="--", alpha=0.8)

ax.set_xlabel("median magnitude")
ax.set_ylabel(r"median $\sigma_F/F$ (mmag), measured")
ax.set_title(
    f"Median photometric scatter vs. magnitude per band -- stable stars "
    f"({MAG_BIN_WIDTH:.1f}-mag bins, error bars = $\\sigma_{{IQR}}$)"
)

handles, labels = ax.get_legend_handles_labels()
handles.append(Line2D([0], [0], color="gray", lw=1.3, ls="--"))
labels.append("mmag_phot (photon-noise floor)")
ax.legend(handles, labels, title="band", ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig(fig, "mmag_median_vs_mag_errorbar_allbands_stablestars")
plt.show()

### 6.4  Dispersion vs. magnitude bin, boxplots per band (2x3 grid)

Same as section 8.5 of `02_ReadLSSTCamSourcesInAllbands.ipynb`: one boxplot
panel per band, x-axis = magnitude bin, y-axis = `mmag_meas`.

In [ ]:
BOXPLOT_YMAX = None  # e.g. 60.0 to fix manually, or None to auto-scale

if BOXPLOT_YMAX is None:
    finite_mmag = df_all["mmag_meas"].dropna()
    ymax_common = np.nanpercentile(finite_mmag, 99.0) * 1.2 if len(finite_mmag) > 0 else 50.0
else:
    ymax_common = BOXPLOT_YMAX

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)

median_props = dict(color="black", linewidth=1.5)

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue

    sub_band = df_all.loc[df_all["band"] == band]

    flier_props = dict(
        marker="o",
        markersize=7,
        markerfacecolor="none",
        markeredgecolor=BANDS_COLOR[band],
        markeredgewidth=1.3,
        alpha=0.7,
    )

    box_data, box_labels = [], []
    for k in range(n_mag_bins):
        lo, hi = mag_edges[k], mag_edges[k + 1]
        last_bin = k == n_mag_bins - 1
        sel = sub_band["mag_median"].between(lo, hi, inclusive="both" if last_bin else "left")
        disp = sub_band.loc[sel, "mmag_meas"].dropna().to_numpy()
        disp = disp[np.isfinite(disp) & (disp > 0)]
        if len(disp) < MIN_PTS_PER_BIN:
            continue
        box_data.append(disp)
        box_labels.append(mag_bin_labels[k])

    if len(box_data) == 0:
        ax.set_title(f"band {band} (no data)")
        continue

    bp = ax.boxplot(
        box_data,
        tick_labels=box_labels,
        showfliers=True,
        flierprops=flier_props,
        medianprops=median_props,
        patch_artist=True,
        widths=0.8,
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(BANDS_COLOR[band])
        patch.set_alpha(0.5)

    for idx, disp in enumerate(box_data, start=1):
        ax.text(
            idx,
            0.97 * ymax_common,
            f"n={len(disp)}",
            ha="center",
            va="top",
            fontsize=10,
            color="darkgray",
        )

    n_total = sum(len(d) for d in box_data)
    ax.set_title(f"band {band}  (N={n_total})")
    ax.set_ylim(0, ymax_common)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.2)

for ax in axes[1, :]:
    ax.set_xlabel("magnitude bin")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured")

fig.suptitle(
    "Photometric scatter vs. magnitude bin per band -- boxplots, stable stars "
    f"(outliers = open circles, common y-scale up to {ymax_common:.0f} mmag)",
    y=1.0,
    fontsize=16,
)
plt.tight_layout()
savefig(fig, "mmag_boxplot_vs_magbin_per_band_stablestars")
plt.show()

### 6.5  Dispersion vs. magnitude bin, violin plots per band, log y-axis
(2x3 grid)

Same as section 8.6 of `02_ReadLSSTCamSourcesInAllbands.ipynb`: violin
plots (full distribution shape) computed on `log10(mmag_meas)`, one panel
per band, with the photon-noise floor (median `mmag_phot` in bin) marked as
a horizontal tick per violin.

In [ ]:
VIOLIN_YMIN = None  # e.g. 2.0 to fix manually
VIOLIN_YMAX = None  # e.g. 500.0 to fix manually

_all_disp = df_all["mmag_meas"].dropna().to_numpy()
_all_disp = _all_disp[np.isfinite(_all_disp) & (_all_disp > 0)]

if VIOLIN_YMIN is None:
    yvmin_common = max(np.nanpercentile(_all_disp, 0.5) * 0.8, _all_disp.min()) if len(_all_disp) else 1.0
else:
    yvmin_common = VIOLIN_YMIN

if VIOLIN_YMAX is None:
    yvmax_common = np.nanpercentile(_all_disp, 99.5) * 1.2 if len(_all_disp) else 500.0
else:
    yvmax_common = VIOLIN_YMAX

log_yvmin, log_yvmax = np.log10(yvmin_common), np.log10(yvmax_common)

_candidate_ticks = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
tick_vals = [v for v in _candidate_ticks if yvmin_common <= v <= yvmax_common]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)

for ax, band in zip(axes.flat, band_grid_order):
    if band not in df_all["band"].unique():
        ax.set_title(f"band {band} (no data)")
        continue

    sub_band = df_all.loc[df_all["band"] == band]

    box_data, box_labels, phot_thresholds = [], [], []
    for k in range(n_mag_bins):
        lo, hi = mag_edges[k], mag_edges[k + 1]
        last_bin = k == n_mag_bins - 1
        sel = sub_band["mag_median"].between(lo, hi, inclusive="both" if last_bin else "left")
        disp = sub_band.loc[sel, "mmag_meas"].dropna().to_numpy()
        disp = disp[np.isfinite(disp) & (disp > 0)]
        if len(disp) < MIN_PTS_PER_BIN:
            continue
        box_data.append(disp)
        box_labels.append(mag_bin_labels[k])

        phot = sub_band.loc[sel, "mmag_phot"].dropna().to_numpy()
        phot = phot[np.isfinite(phot) & (phot > 0)]
        phot_thresholds.append(np.median(phot) if len(phot) > 0 else np.nan)

    if len(box_data) == 0:
        ax.set_title(f"band {band} (no data)")
        continue

    log_data = [np.log10(d) for d in box_data]
    positions = np.arange(1, len(log_data) + 1)

    # violinplot needs at least 2 points per group; fall back to a scatter
    # marker for any singleton bin.
    multi_idx = [i for i, d in enumerate(log_data) if len(d) >= 2]
    if multi_idx:
        vp = ax.violinplot(
            [log_data[i] for i in multi_idx],
            positions=positions[multi_idx],
            widths=0.8,
            showmedians=True,
            showextrema=True,
        )
        for body in vp["bodies"]:
            body.set_facecolor(BANDS_COLOR[band])
            body.set_edgecolor(BANDS_COLOR[band])
            body.set_alpha(0.5)
        for key in ("cmedians", "cmins", "cmaxes", "cbars"):
            vp[key].set_edgecolor(BANDS_COLOR[band])
            vp[key].set_linewidth(1.2)

    single_idx = [i for i, d in enumerate(log_data) if len(d) == 1]
    if single_idx:
        ax.scatter(
            positions[single_idx],
            [log_data[i][0] for i in single_idx],
            marker="D",
            s=40,
            color=BANDS_COLOR[band],
            alpha=0.8,
            zorder=4,
        )

    phot_thresholds_arr = np.asarray(phot_thresholds, dtype=float)
    valid_phot = np.isfinite(phot_thresholds_arr) & (phot_thresholds_arr > 0)
    ax.scatter(
        positions[valid_phot],
        np.log10(phot_thresholds_arr[valid_phot]),
        marker="_",
        s=400,
        linewidths=2.5,
        color="black",
        zorder=5,
        label="mmag_phot (photon-noise floor, median in bin)",
    )

    y_text = log_yvmax - 0.03 * (log_yvmax - log_yvmin)
    for idx, disp in zip(positions, box_data):
        ax.text(
            idx,
            y_text,
            f"n={len(disp)}",
            ha="center",
            va="top",
            fontsize=10,
            color="gray",
        )

    n_total = sum(len(d) for d in box_data)
    ax.set_title(f"band {band}  (N={n_total})")
    ax.set_xticks(positions)
    ax.set_xticklabels(box_labels)
    ax.set_ylim(log_yvmin, log_yvmax)
    ax.set_yticks(np.log10(tick_vals))
    ax.set_yticklabels([str(v) for v in tick_vals])
    ax.tick_params(axis="x", rotation=45)
    ax.grid(True, alpha=0.2)
    ax.legend(loc="lower right", fontsize=7, frameon=False)

for ax in axes[1, :]:
    ax.set_xlabel("magnitude bin")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\sigma_F/F$ (mmag), measured -- log scale")

fig.suptitle(
    "Photometric scatter vs. magnitude bin per band -- violin plots, log-binned, stable stars "
    f"(shared y-range {yvmin_common:.1f}-{yvmax_common:.0f} mmag; diamond = single-point bin)",
    y=1.0,
    fontsize=16,
)
plt.tight_layout()
savefig(fig, "mmag_violin_vs_magbin_per_band_logy_stablestars")
plt.show()

## 7. Summary table export

Save `df_all` (the per-(star, band) summary used throughout this notebook)
next to the figures, for reuse in later notebooks (e.g. once the larger
200+-star sample from `05b_FindSourcesAllSpectypes/usdf_butler` is ready and
this notebook is re-pointed at it).

In [ ]:
out_csv = os.path.join(DIR_FIGS, "objectstats_stablestars_allbands.csv")
df_all.to_csv(out_csv, index=False)
log.info("Summary table saved: %s (%d rows)", out_csv, len(df_all))